# nn-module-subclass composite — cx22: custom Linear: y = x @ W.T + b inside a subclass

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `nn-module-subclass`, `linear-affine-on-custom-tensor`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "nn-module-subclass"
DD_ATOM_IDS = ["nn-module-subclass", "linear-affine-on-custom-tensor"]
DD_SUBTOPICS = ["PyTorch: nn.Module subclassing", "Backprop: Linear affine on custom Tensor"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's `Linear` re-implementation is the canonical first nn.Module subclass exercise. Two atoms running together:
- **nn-module-subclass** — `class Linear(nn.Module)` + `super().__init__()` + `forward`.
- **linear-affine-on-custom-tensor** — the math: `y = x @ W.T + b`, where `W` has shape `(out_features, in_features)` and `b` has shape `(out_features,)`. Both are wrapped as `nn.Parameter` so they show up in `.parameters()`.

**Why `W.T`?** PyTorch stores `W` as `(out_features, in_features)` (rows = output neurons). The forward computes `x @ W.T` so the `(B, in_features) @ (in_features, out_features)` matmul produces `(B, out_features)`. This is the SAME shape convention `nn.Linear` uses.

**Bias broadcast.** `b` has shape `(out_features,)`. Adding to `(B, out_features)` broadcasts across batch — one bias vector per output neuron, shared across the batch.

**Anatomy.**
1. `super().__init__()` + store `in_features`, `out_features` on `self`.
2. Kaiming-init `W` of shape `(out_features, in_features)`; uniform-init `b` of shape `(out_features,)` in `[-1/sqrt(in_features), 1/sqrt(in_features)]` (matches `nn.Linear`).
3. Wrap both as `nn.Parameter`.
4. `forward(x): return x @ self.weight.T + self.bias`.

### Composite Exercise — custom Linear: y = x @ W.T + b inside a subclass

**Atoms exercised together**: `nn-module-subclass`, `linear-affine-on-custom-tensor`

Define a class `MyAffine(nn.Module)` and a builder `cx22_build_affine(in_features, out_features)`.

`MyAffine.__init__` must:
1. `super().__init__()`.
2. Store `self.in_features`, `self.out_features`.
3. Create `weight` of shape `(out_features, in_features)`, Kaiming-init (`nn.init.kaiming_uniform_(w, a=5**0.5)`), wrap as `nn.Parameter`, assign to `self.weight`.
4. Create `bias` of shape `(out_features,)`, uniform-init in `[-1/sqrt(in_features), 1/sqrt(in_features)]` (use `nn.init.uniform_(b, -bound, bound)` with `bound = 1 / in_features**0.5`), wrap as `nn.Parameter`, assign to `self.bias`.

`MyAffine.forward(self, x)` returns `x @ self.weight.T + self.bias`.

Shape contract: `x` is `(..., in_features)`. Output is `(..., out_features)` — the affine applies along the LAST axis, broadcasting across any leading batch dims.

In [ ]:
class MyAffine(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        # Weight: (out_features, in_features). Kaiming-init matches nn.Linear default.
        w = t.empty(out_features, in_features)
        nn.init.kaiming_uniform_(w, a=5 ** 0.5)
        self.weight = nn.Parameter(w)
        # Bias: uniform in [-1/sqrt(in), 1/sqrt(in)] — also matches nn.Linear default.
        bound = 1.0 / (in_features ** 0.5)
        b = t.empty(out_features)
        nn.init.uniform_(b, -bound, bound)
        self.bias = nn.Parameter(b)

    def forward(self, x):
        # Atom (linear-affine-on-custom-tensor): y = x @ W.T + b.
        # x: (..., in_features). W.T: (in_features, out_features). y: (..., out_features).
        # bias broadcasts on the last axis.
        return x @ self.weight.T + self.bias


def cx22_build_affine(in_features: int, out_features: int) -> 'MyAffine':
    return MyAffine(in_features, out_features)


<details><summary>Show solution — cx22</summary>

```python
class MyAffine(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        # Weight: (out_features, in_features). Kaiming-init matches nn.Linear default.
        w = t.empty(out_features, in_features)
        nn.init.kaiming_uniform_(w, a=5 ** 0.5)
        self.weight = nn.Parameter(w)
        # Bias: uniform in [-1/sqrt(in), 1/sqrt(in)] — also matches nn.Linear default.
        bound = 1.0 / (in_features ** 0.5)
        b = t.empty(out_features)
        nn.init.uniform_(b, -bound, bound)
        self.bias = nn.Parameter(b)

    def forward(self, x):
        # Atom (linear-affine-on-custom-tensor): y = x @ W.T + b.
        # x: (..., in_features). W.T: (in_features, out_features). y: (..., out_features).
        # bias broadcasts on the last axis.
        return x @ self.weight.T + self.bias


def cx22_build_affine(in_features: int, out_features: int) -> 'MyAffine':
    return MyAffine(in_features, out_features)
```

The `.T` is the only "trick". Storing `W` as `(out, in)` is PyTorch convention because it lines up with how rows of a weight matrix correspond to output neurons. The matmul `x @ W.T` is what makes the shapes work; if you forget `.T`, you'll see a shape mismatch error pointing at the matmul.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx22'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx22',
        'subtopics': ["PyTorch: nn.Module subclassing", "Backprop: Linear affine on custom Tensor"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()